# `CyMirror` はシリンドリカルに振る舞っているか

シリンドリカルミラーは、片方の面内では集光し、もう片方では何もしない。このノートブックは `CyMirror` にガウシアンビームを入射させ、gtrace が計算する反射光・透過光の q パラメータが理論どおりかを確かめる。

理論は Siegman, *Lasers*, 15章 Table 15.1 — 任意の入射角における曲面の光線行列。以下では gtrace から import せずに書き下している。比べたいのは **gtrace と本** であって、gtrace と gtrace ではないため。

先に結論を書いておくと、**いまは合っている。以前は合っていなかった。** かつての `CyMirror` は形だけシリンドリカルだった。2つの hit メソッドが「トレース面が見る断面の形」と「面の光学的パワー」を1つの変数に載せていて、曲率がトレース面の外を向いているときその変数は 0 でなければならない（断面は本当に直線になる）ので、パワーまで一緒に 0 になっていた。結果、曲率が面内にあるときは **両方** の面内で集光し、面外にあるときは **どちらでも** 集光しない鏡になっていた。

In [1]:
import unicodedata

import numpy as np

import gtrace.beam as beam
import gtrace.optcomp as opt
import gtrace.optics.gaussian as gauss
from gtrace.optics.geometric import cyl_refl_defl_angle
from gtrace.unit import *

pi = np.pi

WL = 1064*nm
np.set_printoptions(precision=6, suppress=True)

def width(s):
    """表示幅。全角は2桁として数える。"""
    return sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in s)

def pad(s, n):
    """表示幅 n の左寄せ。

    書式の %-12s は文字数で詰めるので、全角を混ぜると表の桁がずれる。
    """
    return s + ' '*max(0, n - width(s))

def rpad(s, n):
    """表示幅 n の右寄せ。数値の列見出しに使う。"""
    return ' '*max(0, n - width(s)) + s

## 理論

半径 $R$ の面に入射角 $\theta_1$ で入り、屈折率 $n_1$ から $n_2$ へ抜けるとする（$\theta_2$ はスネルの法則から）。Table 15.1 は、行列式がすべて 1 になる reduced slope（換算傾角）の規約で、次を与える。

**反射 (d)** — 同じ面でも、入射面内では入射面外より強く集光する。面内では実効的な半径が $R\cos\theta$、面外では $R/\cos\theta$ に見えるため:

$$
M^{\rm refl}_x = \begin{pmatrix} 1 & 0 \\ -\dfrac{2n_1}{R\cos\theta_1} & 1\end{pmatrix}
\qquad
M^{\rm refl}_y = \begin{pmatrix} 1 & 0 \\ -\dfrac{2n_1\cos\theta_1}{R} & 1\end{pmatrix}
$$

**屈折 (f) は入射面内、(g) は入射面外**:

$$
M^{\rm refr}_x = \begin{pmatrix} \dfrac{\cos\theta_2}{\cos\theta_1} & 0 \\[2mm]
\dfrac{n_2\cos\theta_2-n_1\cos\theta_1}{R\cos\theta_1\cos\theta_2} & \dfrac{\cos\theta_1}{\cos\theta_2}\end{pmatrix}
\qquad
M^{\rm refr}_y = \begin{pmatrix} 1 & 0 \\ \dfrac{n_2\cos\theta_2-n_1\cos\theta_1}{R} & 1\end{pmatrix}
$$

**シリンダ** とは、片方の面内には $R$ を見せ、もう片方にはまったく曲率を見せない曲面のことである。つまり、その面内の式で $1/R \to 0$ とすればよい。Siegman も p.616 でそう述べている — シリンドリカルレンズは、曲がっていない座標に対しては集光も偏向も与えない。

ここで **述べていないこと** に注意。曲率のない面内の行列が単位行列になるのは **反射のときだけ** で、それも出射角が入射角に等しいからにすぎない。**屈折では単位行列にならない**。傾いた平面でも $A=\cos\theta_2/\cos\theta_1$、$D=\cos\theta_1/\cos\theta_2$ は残る。界面の前後でビームの幅が変わるからである。

In [2]:
def siegman(theta1, n1, n2, invROC_x, invROC_y):
    """Table 15.1。面内・面外それぞれが見る曲率を与える。"""
    c1 = np.cos(theta1)
    theta2 = np.arcsin(n1*np.sin(theta1)/n2)
    c2 = np.cos(theta2)
    Mrx = np.array([[1., 0.], [-2*n1*invROC_x/c1, 1.]])                 # (d)
    Mry = np.array([[1., 0.], [-2*n1*invROC_y*c1, 1.]])                 # (d)
    Mtx = np.array([[c2/c1, 0.],                                        # (f)
                    [(n2*c2 - n1*c1)*invROC_x/(c1*c2), c1/c2]])
    Mty = np.array([[1., 0.], [(n2*c2 - n1*c1)*invROC_y, 1.]])          # (g)
    return Mrx, Mry, Mtx, Mty

## 面の行列

gtrace の `cyl_refl_defl_angle` は、曲がっている側の曲率と、それがどちらの面内のものかを受け取る。半径 2 m の面に 45 度で入射してガラスに入る場合について、表と突き合わせる。

In [3]:
theta, n1, n2, R = np.deg2rad(45.), 1.0, 1.45, 2.0

def surfaces(curve_direction):
    # 法線はビームの方を向く。ビームは +x 方向に走る。
    r = cyl_refl_defl_angle(0.0, pi - theta, n1, n2, invROC=1./R,
                            curve_direction=curve_direction)
    return r[2:]

for cd, ix, iy in [('h', 1./R, 0.0), ('v', 0.0, 1./R)]:
    got = surfaces(cd)
    want = siegman(theta, n1, n2, ix, iy)
    print("curve_direction = '%s'" % cd)
    for name, g, t in zip(['Mrx', 'Mry', 'Mtx', 'Mty'], got, want):
        print('  %s  gtrace %s   理論 %s   一致 %s'
              % (name, pad(np.array2string(g.ravel()), 42),
                 pad(np.array2string(t.ravel()), 42),
                 np.allclose(g, t, rtol=0, atol=1e-15)))
    print()

curve_direction = 'h'
  Mrx  gtrace [ 1.        0.       -1.414214  1.      ]    理論 [ 1.        0.       -1.414214  1.      ]    一致 True
  Mry  gtrace [ 1.  0. -0.  1.]                            理論 [ 1.  0. -0.  1.]                            一致 True
  Mtx  gtrace [1.234656 0.       0.452589 0.809942]        理論 [1.234656 0.       0.452589 0.809942]        一致 True
  Mty  gtrace [1. 0. 0. 1.]                                理論 [1. 0. 0. 1.]                                一致 True

curve_direction = 'v'
  Mrx  gtrace [ 1.  0. -0.  1.]                            理論 [ 1.  0. -0.  1.]                            一致 True
  Mry  gtrace [ 1.        0.       -0.707107  1.      ]    理論 [ 1.        0.       -0.707107  1.      ]    一致 True
  Mtx  gtrace [1.234656 0.       0.       0.809942]        理論 [1.234656 0.       0.       0.809942]        一致 True
  Mty  gtrace [1.       0.       0.279396 1.      ]        理論 [1.       0.       0.279396 1.      ]        一致 True



`'v'` の行をよく見てほしい。`Mrx` は単位行列 — 曲率のない面内での反射 — だが、`Mtx` は **そうではない**。失われたのは C 要素だけである。残っているのは傾きによるスケーリングで、反射と屈折のうち取り違えやすいのはこの屈折のほうである。

## 鏡にビームを入れる

原点に 1 mm のウエスト、1 m の自由空間、その先に半径 2 m のシリンドリカルミラーを 45 度で置く。鏡は3つ用意する — 比較用の `Mirror` と、向きを変えた2つの `CyMirror`。

In [4]:
Q0 = gauss.Rw2q(np.inf, 1*mm)

def probe():
    return beam.GaussianBeam(q0=Q0, wl=WL, pos=[0.0, 0.0], dirAngle=0.0)

def make(cls, theta, **kw):
    return cls(HRcenter=[1.0, 0], normAngleHR=pi - theta,
               diameter=10*cm, thickness=2*cm, wedgeAngle=0.0,
               inv_ROC_HR=1./R, inv_ROC_AR=0.0,
               Refl_HR=0.5, Trans_HR=0.5, Refl_AR=0.5, Trans_AR=0.5,
               n=1.45, name='M', **kw)

sph = make(opt.Mirror, theta)
cyh = make(opt.CyMirror, theta, curve_direction='h')
cyv = make(opt.CyMirror, theta, curve_direction='v')

# 3つとも頂点が [1, 0] にあるので、ビームは同じ 1 m を進んで到達する。
# 違うのは行列だけ。
at_mirror = probe()
at_mirror.propagate(1.0)
print('鏡に到達したときのビーム')
print('  q  = %s' % at_mirror.qx)
print('  w  = %.6f mm    R = %.6f m'
      % (gauss.q2w(at_mirror.qx, WL)/mm, gauss.q2R(at_mirror.qx)))

鏡に到達したときのビーム
  q  = (1+2.952624674426497j)
  w  = 1.055796 mm    R = 9.717992 m


### 反射

In [5]:
rows = []
for label, m in [('Mirror', sph), ("CyMirror 'h'", cyh), ("CyMirror 'v'", cyv)]:
    r = m.hitFromHR(probe())['r1']
    rows.append((label, r.qx, r.qy))

print('%s %s %s' % (pad('', 14), pad('トレース面内の q (x)', 42),
                    pad('トレース面外の q (y)', 42)))
for label, qx, qy in rows:
    print('%s %s %s' % (pad(label, 14), pad(str(qx), 42), pad(str(qy), 42)))
print()
for text, ok in [
        ('CyMirror h は x を球面と同じだけ集光する',
         np.isclose(rows[1][1], rows[0][1], rtol=0, atol=1e-15)),
        ('CyMirror v は y を球面と同じだけ集光する',
         np.isclose(rows[2][2], rows[0][2], rtol=0, atol=1e-15)),
        ('CyMirror h は y を到達時のまま返す',
         np.isclose(rows[1][2], at_mirror.qy, rtol=0, atol=1e-15)),
        ('CyMirror v は x を到達時のまま返す',
         np.isclose(rows[2][1], at_mirror.qx, rtol=0, atol=1e-15))]:
    print('%s : %s' % (pad(text, 44), ok))

               トレース面内の q (x)                       トレース面外の q (y)                      
Mirror         (-0.7237412981338769+0.16769075564406474j) (-1.3210226027755174+0.6642899985332854j) 
CyMirror 'h'   (-0.7237412981338769+0.16769075564406474j) (0.9999999999999999+2.952624674426497j)   
CyMirror 'v'   (1+2.952624674426497j)                     (-1.3210226027755174+0.6642899985332854j) 

CyMirror h は x を球面と同じだけ集光する     : True
CyMirror v は y を球面と同じだけ集光する     : True
CyMirror h は y を到達時のまま返す           : True
CyMirror v は x を到達時のまま返す           : True


2つのシリンダは、互いにラベルを貼り替えただけのものではない。`'h'` の x と `'v'` の y は同じ数にならない。球面が入射面内で入射面外より強く集光するという性質を、シリンダもそのまま受け継ぐからである。

### 焦点距離と $\cos^2\theta$

薄い素子の前後では $1/q' - 1/q = -1/f$ が成り立つ。面内の焦点距離は $R\cos\theta/2$、面外は $R/(2\cos\theta)$ なので、両者は $\cos^2\theta$ だけ違う — 45 度なら2倍である。

In [6]:
def focal(before, after):
    return -1.0/((1.0/after) - (1.0/before)).real

print('%6s  %12s  %12s  %s  %10s'
      % ('theta', 'f_h [m]', 'f_v [m]', rpad('比', 10), 'cos^2'))
for deg in [0., 15., 30., 45., 60., 75.]:
    t = np.deg2rad(deg)
    a = probe()
    a.propagate(1.0)
    h = make(opt.CyMirror, t, curve_direction='h').hitFromHR(probe())['r1']
    v = make(opt.CyMirror, t, curve_direction='v').hitFromHR(probe())['r1']
    f_h = focal(a.qx, h.qx)
    f_v = focal(a.qy, v.qy)
    print('%5.1f   %12.9f  %12.9f  %10.7f  %10.7f'
          % (deg, f_h, f_v, f_h/f_v, np.cos(t)**2))
print()
print('理論: f_h = R cos(t)/2,  f_v = R/(2 cos(t))')

 theta       f_h [m]       f_v [m]          比       cos^2
  0.0    1.000000000   1.000000000   1.0000000   1.0000000
 15.0    0.965925826   1.035276180   0.9330127   0.9330127
 30.0    0.866025404   1.154700538   0.7500000   0.7500000
 45.0    0.707106781   1.414213562   0.5000000   0.5000000
 60.0    0.500000000   2.000000000   0.2500000   0.2500000
 75.0    0.258819045   3.863703305   0.0669873   0.0669873

理論: f_h = R cos(t)/2,  f_v = R/(2 cos(t))


### 鏡がビームに何をするか

「シリンドリカルである」ことが最もはっきり出るのは、下流で何が起きるかである。45 度の `'h'` 鏡にビームを当て、面内・面外それぞれのビーム半径を追ってみる。片方はウエストを結び、もう片方は何も無かったかのように広がり続ける。

In [7]:
r = cyh.hitFromHR(probe())['r1']

print('%8s  %10s  %10s' % ('z [m]', 'w_x [mm]', 'w_y [mm]'))
print('%8s  %s  %s' % ('', rpad('(集光する)', 10), rpad('(素通り)', 10)))
for z in np.arange(0.0, 1.61, 0.1):
    b = r.copy()
    b.propagate(z)
    wx = gauss.q2w(b.qx, WL)/mm
    wy = gauss.q2w(b.qy, WL)/mm
    bar = '#'*int(round(wx*12)) + ' '*40
    print('%8.2f  %10.6f  %10.6f  %s' % (z, wx, wy, bar[:34]))

   z [m]    w_x [mm]    w_y [mm]
          (集光する)    (素通り)
    0.00    1.055796    1.055796  #############                     
    0.10    0.917909    1.067143  ###########                       
    0.20    0.781538    1.079433  #########                         
    0.30    0.647643    1.092635  ########                          
    0.40    0.518144    1.106717  ######                            
    0.50    0.397365    1.121645  #####                             
    0.60    0.296174    1.137385  ####                              
    0.70    0.240691    1.153906  ###                               
    0.80    0.261800    1.171173  ###                               
    0.90    0.345745    1.189154  ####                              
    1.00    0.459275    1.207817  ######                            
    1.10    0.585424    1.227131  #######                           
    1.20    0.717567    1.247066  #########                         
    1.30    0.852923    1.267593  ##########

In [8]:
# 対照として、同じビームを球面鏡で反射させる。こちらは面内・面外の
# 両方が、少しずれた位置でそれぞれ集光する — 非点収差であって、
# シリンドリカルではない。
s = sph.hitFromHR(probe())['r1']
print('%8s  %10s  %10s' % ('z [m]', 'w_x [mm]', 'w_y [mm]'))
for z in np.arange(0.0, 1.61, 0.1):
    b = s.copy()
    b.propagate(z)
    print('%8.2f  %10.6f  %10.6f'
          % (z, gauss.q2w(b.qx, WL)/mm, gauss.q2w(b.qy, WL)/mm))

   z [m]    w_x [mm]    w_y [mm]
    0.00    1.055796    1.055796
    0.10    0.917909    0.992523
    0.20    0.781538    0.930427
    0.30    0.647643    0.869761
    0.40    0.518144    0.810846
    0.50    0.397365    0.754092
    0.60    0.296174    0.700025
    0.70    0.240691    0.649317
    0.80    0.261800    0.602815
    0.90    0.345745    0.561567
    1.00    0.459275    0.526806
    1.10    0.585424    0.499889
    1.20    0.717567    0.482131
    1.30    0.852923    0.474561
    1.40    0.990175    0.477664
    1.50    1.128631    0.491238
    1.60    1.267897    0.514454


## 透過

基板を通り抜けるとき、曲がっていない側の面内では、面のパワーだけが失われ、それ以外はすべて保たれなければならない — 屈折率の変化と、傾きによるスケーリングである。行列が対角なら、gtrace の q はちょうど $(n_2/n_1)A^2$ 倍される。これを使えば、この主張は最後の1ビットまで確かめられる。

In [9]:
N = 1.45
theta2 = np.arcsin(np.sin(theta)/N)

for cd in ['h', 'v']:
    m = make(opt.CyMirror, theta, curve_direction=cd)
    bs = m.hitFromHR(probe(), order=2)
    a = probe()
    a.propagate(bs['input'].length)
    inside = bs['s1'].copy()
    inside.propagate(bs['s1'].length)

    # シリンダが曲げていない側の面内
    was      = a.qy       if cd == 'h' else a.qx
    in_glass = bs['s1'].qy if cd == 'h' else bs['s1'].qx
    at_ar    = inside.qy  if cd == 'h' else inside.qx
    out      = bs['t1'].qy if cd == 'h' else bs['t1'].qx

    A_in  = 1.0 if cd == 'h' else np.cos(theta2)/np.cos(theta)
    A_out = 1.0 if cd == 'h' else np.cos(theta)/np.cos(theta2)

    print("curve_direction = '%s' のとき、曲率を持たないのは %s 面内"
          % (cd, 'y' if cd == 'h' else 'x'))
    print('  %s : 比 %.12f   n*A^2 = %.12f   %s'
          % (pad('入るとき', 8), (in_glass/was).real, N*A_in**2,
             np.isclose(in_glass, N*A_in**2*was, rtol=0, atol=1e-14)))
    print('  %s : 比 %.12f   A^2/n = %.12f   %s'
          % (pad('出るとき', 8), (out/at_ar).real, A_out**2/N,
             np.isclose(out, A_out**2/N*at_ar, rtol=0, atol=1e-14)))
    print('  %s : %s'
          % (pad('端から端まで、1/R はどこにも現れない', 44),
             np.isclose(out, A_out**2/N*(N*A_in**2*was + bs['s1'].length),
                        rtol=0, atol=1e-14)))
    print()

curve_direction = 'h' のとき、曲率を持たないのは y 面内
  入るとき : 比 1.450000000000   n*A^2 = 1.450000000000   True
  出るとき : 比 0.689655172414   A^2/n = 0.689655172414   True
  端から端まで、1/R はどこにも現れない         : True

curve_direction = 'v' のとき、曲率を持たないのは x 面内
  入るとき : 比 2.210344827586   n*A^2 = 2.210344827586   True
  出るとき : 比 0.452418096724   A^2/n = 0.452418096724   True
  端から端まで、1/R はどこにも現れない         : True



## まとめ

* 面の行列は Table 15.1 そのもので、曲率は片方の面内にだけ与えられ、もう片方には 0 が与えられる。
* 曲率のない面内の行列は、反射では単位行列になるが、屈折ではならない。A と D は残る。
* 面内と面外の焦点距離は、そうあるべきとおり $\cos^2\theta$ だけ違う。
* シリンドリカルミラーは片方の面内をウエストに導き、もう片方は手をつけずに通す。

同じ検査は `tests/gui/verify_cylindrical.py` で件数付きのアサーションとして走り、`tests/gui/run_all.py` がそれを回している。したがって、このノートブックは読み物であって、正であるのはあちらのほうである。

英語版は `tests/cymirror_verification.ipynb`。